### Cell 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import os

# Setting plotting styles
plt.style.use('fivethirtyeight')
sns.set_context("talk")
%matplotlib inline

print("Imports successful.")

### Cell 2: Load CSV and Convert to Parquet

In [ ]:
raw_path = "<RAW_DATA_DIR>/trip_positions.csv"
processed_path = "<PROCESSED_DATA_DIR>/trip_positions.parquet"

print(f"Loading {raw_path}...")
df = pd.read_csv(raw_path)

print(f"Shape: {df.shape}")
print("Dtypes:")
print(df.dtypes)

print(f"Saving to {processed_path} using pyarrow engine...")
df.to_parquet(processed_path, engine='pyarrow', index=False)

print(f"Reloading from {processed_path}...")
df = pd.read_parquet(processed_path)

print("✅ CSV successfully converted to Parquet and reloaded!")

### Cell 3: Basic Health Check

In [ ]:
print("Checking Basic Data Health...")
print(f"Shape: {df.shape}")
print("\nColumn Dtypes:")
print(df.dtypes)

print("\nNull counts per column:")
print(df.isnull().sum())

total_dupes = df.duplicated().sum()
print(f"\nTotal duplicate rows: {total_dupes:,}")

pos_id_dupes = df.duplicated(subset=['position_id']).sum()
print(f"Duplicate position_id count: {pos_id_dupes:,}")

### Cell 4: Coordinate Validity Check
Flag GPS coordinates that fall outside the expected geographic range or are recorded as zero.


In [ ]:
print("Checking coordinate validity...")

lat_min, lat_max = 44.0, 46.0
lng_min, lng_max = -95.0, -92.0

invalid_lat = (df['lat'] < lat_min) | (df['lat'] > lat_max)
invalid_lng = (df['lng'] < lng_min) | (df['lng'] > lng_max)
zero_coords = (df['lat'] == 0) | (df['lng'] == 0)

invalid_mask = invalid_lat | invalid_lng | zero_coords
invalid_rows = df[invalid_mask]

print(f"Rows with invalid latitude:  {invalid_lat.sum():,}")
print(f"Rows with invalid longitude: {invalid_lng.sum():,}")
print(f"Rows with zero coordinates:  {zero_coords.sum():,}")
print(f"Total invalid rows:          {len(invalid_rows):,}")

if not invalid_rows.empty:
    print("\nSample of invalid rows:")
    display(invalid_rows.head())


### Cell 5: Frozen Device Detection
Identify instances where a GPS device repeats identical coordinates across consecutive pings.


In [ ]:
print("Detecting frozen pings (identical coordinates for same vehicle)...")

# Sort for comparison
df = df.sort_values(['vehicle_id', 'timestamp'])

# Check if current lat/lng matches previous lat/lng for the same vehicle
df['is_frozen'] = (df['lat'] == df['lat'].shift(1)) & \
                  (df['lng'] == df['lng'].shift(1)) & \
                  (df['vehicle_id'] == df['vehicle_id'].shift(1))

frozen_counts = df.groupby('vehicle_id')['is_frozen'].sum().sort_values(ascending=False)

print(f"Total frozen pings detected: {df['is_frozen'].sum():,}")

plt.figure(figsize=(12, 6))
frozen_counts.head(10).plot(kind='bar', color='salmon')
plt.title("Top 10 Vehicles with Most Frozen Pings")
plt.xlabel("Vehicle ID")
plt.ylabel("Frozen Ping Count")
plt.xticks(rotation=45)
plt.show()

print("\nTop 10 worst vehicles for frozen pings:")
print(frozen_counts.head(10))


### Cell 6: GPS Gap Detection
Calculate time intervals between pings to identify significant signal losses (gaps > 300s).


In [ ]:
print("Detecting GPS signal gaps...")

# Calculate gaps
df['gap_s'] = df.groupby('vehicle_id')['timestamp'].diff()

print(f"Total gaps > 300 seconds: {(df['gap_s'] > 300).sum():,}")

# Plot distribution
plt.figure(figsize=(12, 6))
sns.histplot(df[df['gap_s'] > 0]['gap_s'], bins=100, color='teal', log_scale=(True, False))
plt.title("Distribution of GPS Signal Gaps (Seconds)")
plt.xlabel("Gap Duration (seconds)")
plt.ylabel("Frequency")
plt.show()

print("\nTop 10 longest gaps:")
print(df.sort_values('gap_s', ascending=False)[['vehicle_id', 'timestamp', 'gap_s']].head(10))

### Cell 7: Timestamp Consistency Check
Verify that the Unix timestamp matches the provided UTC string column.


In [ ]:
print("Checking consistency between 'timestamp' and 'timestamp_utc'...")

# Convert Unix to UTC datetime
df['calculated_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df['original_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)

# Calculate difference
time_diff = (df['calculated_utc'] - df['original_utc']).dt.total_seconds().abs()
mismatches = time_diff > 5

print(f"Total timestamp mismatches (>5s difference): {mismatches.sum():,}")

if mismatches.any():
    print("\nSample of mismatched timestamps:")
    display(df[mismatches][['timestamp', 'timestamp_utc', 'calculated_utc', 'original_utc']].head())


### Cell 8: Pings Per Trip Distribution
Analyze the volume of data per trip and flag "thin" trips with fewer than 5 pings.


In [ ]:
print("Analyzing pings per trip...")

ping_counts = df.groupby('trip_id').size().reset_index(name='ping_count')

print("\nStatistics for pings per trip:")
print(ping_counts['ping_count'].describe())

low_ping_trips = ping_counts[ping_counts['ping_count'] < 5]
print(f"\nTrips with fewer than 5 pings: {len(low_ping_trips):,}")

plt.figure(figsize=(12, 6))
sns.histplot(ping_counts['ping_count'], bins=50, color='purple', log_scale=(False, True))
plt.title("Distribution of Ping Counts per Trip (Log Scale Y)")
plt.xlabel("Number of Pings")
plt.ylabel("Count of Trips")
plt.show()


### Cell 9: Summary Report
Consolidated health metrics for the dataset.

In [ ]:
print("Generating Final Data Health Summary Report...")

summary_data = {
    "Metric": [
        "Total Rows",
        "Duplicate Rows",
        "Invalid Coordinates (Out of Range)",
        "Zero Coordinates (0,0)",
        "Frozen Pings (Identical Coords)",
        "Large Gaps (>300s)",
        "Timestamp Mismatches (>5s)",
        "Low Ping Trips (<5 pings)"
    ],
    "Count": [
        len(df),
        total_dupes,
        invalid_mask.sum(),
        zero_coords.sum(),
        df['is_frozen'].sum(),
        (df['gap_s'] > 300).sum(),
        mismatches.sum(),
        len(low_ping_trips)
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Percentage'] = (summary_df['Count'] / len(df) * 100).round(4).astype(str) + '%'

# Format counts with commas
summary_df['Count'] = summary_df['Count'].apply(lambda x: f"{x:,}")

display(summary_df)

print("\nNull Counts per Column:")
print(df.isnull().sum())

### Cell 10 — Time-Based Validations & Collection Window Checks
Extract specific time components and validate that all pings fall within the study period (Sept 2025 - Feb 2026) and maintain logical chronological order.


In [ ]:
# 1. Convert timestamp_utc to local time
print("Converting to local timezone (America/Chicago)...")
# Note: timestamp_utc must be tz-aware (UTC) for tz_convert to work
df['local_timestamp'] = df['timestamp_utc'].dt.tz_convert('America/Chicago')

# 2. Extract operational date and time from the local timestamp
# This makes analysis much easier as school hours will align with the data
df['date'] = df['local_timestamp'].dt.date
df['time_only'] = df['local_timestamp'].dt.time
df['hour'] = df['local_timestamp'].dt.hour  # Adding hour for easier filtering

# 3. Quick verification
print("-" * 30)
print("Timezone conversion successful.")
sample = df[['timestamp_utc', 'local_timestamp']].iloc[0]
print(f"UTC:   {sample['timestamp_utc']}")
print(f"Local: {sample['local_timestamp']}")
print("-" * 30)

# 4. Optional: Show pings by local hour (Operationally useful)
print("\nPings by Local Hour of Day:")
print(df['hour'].value_counts().sort_index())


In [ ]:
df = df.sort_values(['trip_id', 'timestamp_utc']).reset_index(drop=True)
#  Check for timestamps outside the collection window (Sep 1 2025 – Feb 13 2026)
start_window = pd.Timestamp('2025-09-01', tz='UTC')
end_window = pd.Timestamp('2026-02-13 23:59:59', tz='UTC')

df['is_out_of_window'] = (df['timestamp_utc'] < start_window) | (df['timestamp_utc'] > end_window)
out_of_window_count = df['is_out_of_window'].sum()


# Display Results
print("-" * 30)
print(f"Data Window: {start_window.date()} to {end_window.date()}")
print(f"Pings outside study window: {out_of_window_count:,}")
print(f"Out-of-order pings detected: {out_of_order_count:,}")
print("-" * 30)

# Show statistics for dates
print("\nPings by Date (Top 10):")
print(df['date'].value_counts().sort_index().head(10))

## Trip Sequencing
**Ping Sequencing**: Creates a `ping_sequence` (row 1 to N) for every trip ID. This ensures the data is strictly chronological and provides a continuous index for calculating speeds and distances between subsequent pings.

In [ ]:
# Create ping_sequence: Row number within each trip ordered by timestamp
print("Calculating ping sequences within trips...")

# 1. Ensure the dataframe is sorted chronologically within each trip
# This is required for cumcount() to assign meaningful sequence numbers
df = df.sort_values(by=['trip_id', 'timestamp_utc']).reset_index(drop=True)

# 2. Generate sequence numbers starting at 1 for each trip
df['ping_sequence'] = df.groupby('trip_id').cumcount() + 1

# 3. Verification
print("Ping sequence created successfully.")
sample_trip_id = df['trip_id'].unique()[0]
print(f"\nSample sequence for Trip ID {sample_trip_id}:")
print(df[df['trip_id'] == sample_trip_id][['trip_id', 'ping_sequence', 'local_timestamp']].head(10))

# Quick check on the longest trip (most pings)
max_seq = df['ping_sequence'].max()
print(f"\nMax sequence length found: {max_seq:,} pings in a single trip.")


###  Export Cleaned Dataset
Final data cleaning (filtering out invalid pings) and export to both Parquet (for performance) and CSV (for portability).


In [ ]:
import os
# Drop unnecessary columns
cols_to_drop = ['calculated_utc', 'original_utc', 'hour']
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Successfully dropped: {cols_to_drop}")
print(f"Remaining columns: {df.columns.tolist()}")


# 1. Define paths (relative to notebook directory)
output_dir = '<PROCESSED_DATA_DIR>'
os.makedirs(output_dir, exist_ok=True)

# Using 'cleaned' to indicate this is the full data plus the new validated columns
parquet_path = os.path.join(output_dir, 'trip_positions_cleaned.parquet')
csv_path = os.path.join(output_dir, 'trip_positions_cleaned.csv')

# 2. Perform export of the FULL dataframe
print(f"Exporting full dataset ({len(df):,} rows)...")
df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

print("-" * 30)
print(f"EXPORT SUCCESSFUL")
print(f"Final Shape: {df.shape}")
print(f"Files saved to {output_dir}")
print("-" * 30)

